# Predicting Student Health Risk — v8: Native Missing Values + Prior Correction
### Kaggle Playground Series S6E7

**Recap:**
- v3 LightGBM (tuned, median/mode-imputed): CV 0.94965, leaderboard **0.95011** (current best)
- v5 Ensemble, v6 Feature engineering, v7 Pseudo-labeling: all no real gain or worse

**What changed based on community findings:**
Several independent discussion posts converge on one finding: **pre-imputing missing values
destroys signal**, because the missingness itself is informative (Missing Not At Random — e.g.
high stress correlates with `sleep_duration` being missing). One post reported going from
LB 0.903 to 0.950 purely by switching off pre-imputation and letting the model handle NaN
natively.

**Two changes tested here, each validated against the ORIGINAL v3 approach for a clean comparison:**

1. **No pre-imputation.** Numeric columns keep their real NaNs (LightGBM handles missing
   values natively via learned split directions). Categorical columns also keep real NaNs as a
   pandas `category` dtype with `np.nan` (not filled with the string `"missing"`).
2. **Prior correction vs `class_weight='balanced'`.** Multiple posts report prior correction
   (train unweighted, then divide predicted probabilities by class priors and renormalize)
   outperforms `class_weight='balanced'` for balanced accuracy specifically. We test both and
   compare.

Same tuned hyperparameters as v3, so any score change is attributable to these two changes, not
a different model config.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "/kaggle/input/competitions/playground-series-s6e7"


## 1. Load the data (no imputation this time)

In [2]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

target_col = "health_condition"
feature_cols = [c for c in train.columns if c not in [target_col, "id"]]

X = train[feature_cols].copy()
y = train[target_col].copy()
X_test = test[feature_cols].copy()

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()
print("Numeric:", num_cols)
print("Categorical:", cat_cols)

print("\nMissing value counts (train):")
print(X.isnull().sum())

Numeric: ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']
Categorical: ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']

Missing value counts (train):
sleep_duration             75999
heart_rate                  7833
bmi                        13898
calorie_expenditure        52853
step_count                 13916
exercise_duration           6901
water_intake               43477
diet_type                   6901
stress_level               82811
sleep_quality              58331
physical_activity_level    36621
smoking_alcohol            28582
gender                     21373
dtype: int64


## 2. Set dtypes WITHOUT filling missing values

Numeric columns stay as-is (NaN preserved). Categorical columns become pandas `category` dtype
with real `NaN` for missing entries — LightGBM handles NaN in categoricals natively too, no
need for an explicit "missing" string category this time.

In [3]:
for c in cat_cols:
    all_cats = pd.concat([X[c], X_test[c]]).dropna().astype(str).unique()
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")
    # ensure train/test share identical category sets
    X[c] = X[c].cat.set_categories(all_cats)
    X_test[c] = X_test[c].cat.set_categories(all_cats)

target_encoder = LabelEncoder()
y_enc = target_encoder.fit_transform(y)
class_priors = np.bincount(y_enc) / len(y_enc)
print("Class order:", dict(zip(target_encoder.classes_, range(len(target_encoder.classes_)))))
print("Class priors:", dict(zip(target_encoder.classes_, class_priors)))

Class order: {'at-risk': 0, 'fit': 1, 'unhealthy': 2}
Class priors: {'at-risk': np.float64(0.8586745458550213), 'fit': np.float64(0.057678151192311705), 'unhealthy': np.float64(0.08364730295266691)}


## 3. Train with NATIVE missing values — comparing class_weight vs prior correction

Two variants trained per fold:
- **Variant A**: `class_weight='balanced'` (same as v3's approach, now with native NaN handling)
- **Variant B**: no class weighting, but **prior correction** applied to predicted probabilities
  afterward (divide by class priors, renormalize)

Both use the same tuned hyperparameters from v3.

In [4]:
# Best params from v3's Optuna search (model-config params only; class_weight set per-variant below)
base_params = {
    'n_estimators': 500,
    'learning_rate': 0.03426069502422001,
    'num_leaves': 23,
    'max_depth': 9,
    'min_child_samples': 18,
    'subsample': 0.7293691429332224,
    'colsample_bytree': 0.5962321115660644,
    'reg_alpha': 5.178964936328794e-07,
    'reg_lambda': 7.656695372261228e-07,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
}

def prior_correct(proba, priors):
    corrected = proba / priors
    return corrected / corrected.sum(axis=1, keepdims=True)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_a = []  # class_weight='balanced'
scores_b = []  # unweighted + prior correction
test_preds_a = []
test_preds_b = []
oof_proba_a = np.zeros((len(X), 3))
oof_proba_b = np.zeros((len(X), 3))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_enc)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y_enc[train_idx], y_enc[val_idx]

    # --- Variant A: class_weight='balanced' ---
    model_a = LGBMClassifier(**base_params, class_weight="balanced")
    model_a.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[early_stopping(stopping_rounds=50, verbose=False), log_evaluation(period=0)]
    )
    val_proba_a = model_a.predict_proba(X_val)
    oof_proba_a[val_idx] = val_proba_a
    score_a = balanced_accuracy_score(y_val, np.argmax(val_proba_a, axis=1))
    scores_a.append(score_a)
    test_preds_a.append(model_a.predict_proba(X_test))

    # --- Variant B: unweighted + prior correction ---
    model_b = LGBMClassifier(**base_params)  # no class_weight
    model_b.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[early_stopping(stopping_rounds=50, verbose=False), log_evaluation(period=0)]
    )
    val_proba_b_raw = model_b.predict_proba(X_val)
    val_proba_b = prior_correct(val_proba_b_raw, class_priors)
    oof_proba_b[val_idx] = val_proba_b
    score_b = balanced_accuracy_score(y_val, np.argmax(val_proba_b, axis=1))
    scores_b.append(score_b)
    test_preds_b.append(prior_correct(model_b.predict_proba(X_test), class_priors))

    print(f"Fold {fold+1}:  A (class_weight)={score_a:.5f}   B (prior-correct)={score_b:.5f}")

print(f"\nMean CV — Variant A (class_weight='balanced'):   {np.mean(scores_a):.5f} (+/- {np.std(scores_a):.5f})")
print(f"Mean CV — Variant B (unweighted + prior-correct): {np.mean(scores_b):.5f} (+/- {np.std(scores_b):.5f})")
print(f"\nFor reference, v3 (pre-imputed, class_weight):    0.94965")

Fold 1:  A (class_weight)=0.95046   B (prior-correct)=0.95040
Fold 2:  A (class_weight)=0.95138   B (prior-correct)=0.95183
Fold 3:  A (class_weight)=0.94905   B (prior-correct)=0.94906
Fold 4:  A (class_weight)=0.94918   B (prior-correct)=0.94963
Fold 5:  A (class_weight)=0.94807   B (prior-correct)=0.94831

Mean CV — Variant A (class_weight='balanced'):   0.94963 (+/- 0.00116)
Mean CV — Variant B (unweighted + prior-correct): 0.94985 (+/- 0.00120)

For reference, v3 (pre-imputed, class_weight):    0.94965


## 4. Pick the winning variant and prepare submission

Whichever variant scored higher above is used for the final test predictions.

In [5]:
if np.mean(scores_b) > np.mean(scores_a):
    print("Prior correction (Variant B) wins — using it for final submission.")
    avg_test_proba = np.mean(test_preds_b, axis=0)
    winning_oof = oof_proba_b
    winning_score = np.mean(scores_b)
else:
    print("class_weight='balanced' (Variant A) wins — using it for final submission.")
    avg_test_proba = np.mean(test_preds_a, axis=0)
    winning_oof = oof_proba_a
    winning_score = np.mean(scores_a)

final_preds_enc = np.argmax(avg_test_proba, axis=1)
final_preds = target_encoder.inverse_transform(final_preds_enc)

submission = pd.DataFrame({"id": test["id"], "health_condition": final_preds})
assert list(submission.columns) == list(sample_sub.columns)
assert len(submission) == len(sample_sub)
submission.to_csv("submission.csv", index=False)

np.save("v8_oof_proba.npy", winning_oof)
np.save("v8_test_proba.npy", avg_test_proba)

print(f"\nFinal v8 CV score: {winning_score:.5f}")
print(f"Compare to v3: 0.94965")
print(pd.Series(final_preds).value_counts())
submission.head()

Prior correction (Variant B) wins — using it for final submission.

Final v8 CV score: 0.94985
Compare to v3: 0.94965
at-risk      239611
unhealthy     34359
fit           21783
Name: count, dtype: int64


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


## 5. Sanity check: does the sleep/stress/activity "rule" show the same pattern?

The discussion post found balanced accuracy on the complete-key subset (all 3 of
`sleep_duration`, `stress_level`, `physical_activity_level` present) reaches ~0.97, collapsing
toward random as more of those go missing. Let's verify that pattern exists in our own data —
this is a strong sanity check that we're looking at the same phenomenon they found.

In [6]:
rule_cols = ["sleep_duration", "stress_level", "physical_activity_level"]
missing_mask = train[rule_cols].isnull()

# Build the same 3-bit missing pattern they used
pattern = (
    missing_mask["sleep_duration"].astype(int).astype(str) +
    missing_mask["stress_level"].astype(int).astype(str) +
    missing_mask["physical_activity_level"].astype(int).astype(str)
)

print("Row counts by missing pattern (order: sleep, stress, activity):")
print(pattern.value_counts().sort_index())

print(f"\nComplete-key rows (pattern '000'): {(pattern == '000').sum()} "
      f"({100*(pattern == '000').mean():.1f}% of train)")

Row counts by missing pattern (order: sleep, stress, activity):
000    511675
001     28616
010     69792
011      4006
100     63486
101      3500
110      8514
111       499
Name: count, dtype: int64

Complete-key rows (pattern '000'): 511675 (74.1% of train)
